# Cropy 전체 데이터 Colab 학습

학습 32,513장 · 검증 4,070장 · 9개 클래스. 중복 제거된 공식 분할을 유지합니다.

## 준비
1. Google Drive의 `MyDrive/cropy_full_512` 폴더에 **cropy_full_512.zip**과 **training_code.zip**을 업로드합니다.
2. 이 노트북을 Colab에서 열고 GPU 런타임을 선택합니다.
3. 셀을 위에서부터 실행합니다. 결과는 Drive의 `cropy_runs/full512_resnet50_v1`에 저장됩니다.

전체 이미지를 긴 변 512px 이하, 종횡비 유지, JPEG 품질 95 사본으로 묶었습니다. 학습 입력은 256px입니다. **원본 해상도 학습은 아닙니다.** 원본 JSON의 박스 좌표는 원본 크기 기준이므로 이 패키지는 이미지 분류용입니다.

기존 Cropy 체크포인트를 사용하지 않고 ImageNet 가중치로 시작합니다. 별도 최종 test는 없으며 Validation 지표를 최종 독립 테스트 성능으로 부르지 않습니다.

In [ ]:
from pathlib import Path
import csv, hashlib, json, shutil, subprocess, sys, zipfile
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임을 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
UPLOAD = Path('/content/drive/MyDrive/cropy_full_512')
WORK = Path('/content/cropy_full_512')
RUN = Path('/content/drive/MyDrive/cropy_runs/full512_resnet50_v1')
EPOCHS = 20
BATCH_SIZE = 16
WORK.mkdir(parents=True, exist_ok=True)


## 업로드 검증 및 로컬 디스크 추출
학습 이미지는 Drive에서 매번 읽지 않고 Colab 로컬 디스크에 추출합니다. 세션이 초기화되면 이 셀부터 다시 실행합니다.

In [ ]:
EXPECTED_DATA_SHA256 = '52dea0080285d76b6cbc09c2659c052a18f20ef620d05bca462203e64e6872ee'
EXPECTED_CODE_SHA256 = '8f9ec3ef00132ff4d0b53d58210df28f9857ab5a058550dc200691f2773925e2'

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def unpack(name, expected, dest):
    src = UPLOAD / name
    assert src.is_file(), f'Drive에 업로드 필요: {src}'
    local = WORK / name
    if not local.exists() or digest(local) != expected:
        shutil.copyfile(src, local)
    assert digest(local) == expected, f'업로드 파일 해시 불일치: {name}'
    with zipfile.ZipFile(local) as z:
        required = sum(i.file_size for i in z.infolist())
        assert shutil.disk_usage(WORK).free > required + 2_000_000_000, 'Colab 로컬 저장 공간 부족'
        for item in z.infolist():
            target = (dest / item.filename).resolve()
            assert target.is_relative_to(dest.resolve()), '잘못된 압축 경로'
        z.extractall(dest)

unpack('cropy_full_512.zip', EXPECTED_DATA_SHA256, WORK)
unpack('training_code.zip', EXPECTED_CODE_SHA256, WORK / 'code')
DATA = WORK / 'dataset'
print('추출 완료:', DATA)


## 데이터 전수 확인
이미지·라벨 해시, 분할 수량, 중복 및 클래스 순서를 확인합니다.

In [ ]:
from collections import Counter
from torchvision.datasets import ImageFolder
with (DATA / 'manifest.csv').open(encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))
assert len(rows) == 36583
assert Counter(r['split'] for r in rows) == {'1.Training': 32513, '2.Validation': 4070}
assert len({r['sha256'] for r in rows}) == len(rows)
assert len({r['packaged_sha256'] for r in rows}) == len(rows)
for i, r in enumerate(rows, 1):
    assert digest(DATA / r['image_path']) == r['packaged_sha256'], r['image_path']
    assert digest(DATA / r['label_path']) == r['label_sha256'], r['label_path']
    if i % 5000 == 0: print('검증:', i)
train_ds = ImageFolder(DATA / 'images/train')
val_ds = ImageFolder(DATA / 'images/val')
assert (len(train_ds), len(val_ds)) == (32513, 4070)
assert train_ds.class_to_idx == val_ds.class_to_idx
assert [int(c.split('_')[0]) for c in train_ds.classes] == list(range(9))
print('전체 검증 통과:', train_ds.class_to_idx)


## 본 학습 / 이어 학습
ResNet50 · 256px · 20 epoch · AdamW · 학습률 1e-4 · 클래스 균형 샘플링입니다. 검증은 전체 4,070장을 사용합니다. 최고 검증 Macro F1 모델을 저장합니다.

`last_checkpoint.pth`가 있으면 마지막 완료 epoch부터 자동 재개합니다. 중간에 끊긴 epoch는 다시 실행합니다. 재개 시 EPOCHS/BATCH_SIZE 등 설정을 유지하세요. 설정 변경 실험은 RUN 폴더 이름을 바꿉니다.

매 epoch 모델·옵티마이저·스케줄러·난수 상태를 Drive에 저장합니다. GPU/라이브러리 환경이 바뀌면 비트 단위 재현까지 보장하지 않습니다. 첫 epoch 전 메모리 부족이면 BATCH_SIZE를 8로 낮출 수 있습니다.

In [ ]:
command = [sys.executable, '-u', str(WORK / 'code/train_full_colab.py'),
    '--data-dir', str(DATA / 'images'), '--output-dir', str(RUN),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
    '--num-workers', '2', '--img-size', '256', '--arch', 'resnet50']
if (RUN / 'last_checkpoint.pth').exists():
    command.append('--resume')
print('이어 학습:', '--resume' in command)
subprocess.run(command, check=True)


## 결과 확인
정확도뿐 아니라 Macro F1과 질병별 재현율을 확인합니다. 무노균병은 검증 표본이 24장이므로 소수 샘플의 변동에 주의하세요.

In [ ]:
import matplotlib.pyplot as plt
history = json.loads((RUN / 'history.json').read_text())
metrics = json.loads((RUN / 'val_metrics.json').read_text())
config = json.loads((RUN / 'config.json').read_text())
print('최고 모델 검증 정확도:', round(metrics['accuracy'] * 100, 2), '%')
print('최고 모델 검증 Macro F1:', round(metrics['macro_f1'], 4))
for name, recall, support in zip(config['classes'], metrics['recall'], metrics['support']):
    print(name, '재현율:', round(recall * 100, 2), '%', '표본:', support)
plt.plot([r['epoch'] for r in history], [r['val']['macro_f1'] for r in history], label='Validation Macro F1')
plt.xlabel('Epoch'); plt.legend(); plt.grid(); plt.show()
print('결과 저장 위치:', RUN)


## 모델 다운로드 (선택)
Drive에 이미 저장돼 있습니다. 이 파일을 내려받아 별도 검증 후 서비스 적용 여부를 판단하세요. 현재 서비스 모델은 자동으로 교체되지 않습니다.

In [ ]:
from google.colab import files
files.download(str(RUN / 'best_model.pth'))
